# Chaldene Slider Demo

A Python `interact` slider controls the lower threshold bound of a VP cell.
`ChaldeneClient` communicates with the extension over a Jupyter comm channel,
so `set_input` and `run` are ordinary Python calls — no `display(Javascript)` needed.

> Run cells in order with Shift+Enter.

In [1]:
from PIL import Image
import numpy as np

arr = np.random.default_rng(42).integers(0, 256, (256, 256), dtype=np.uint8)
Image.fromarray(arr, mode='L').save('sample.png')
print('sample.png ready (256x256 grayscale)')


sample.png ready (256x256 grayscale)


---
## Step 1 — The VP pipeline

Run this cell to open the visual canvas.

In [ ]:
{"nodes":[{"id":"0","type":"read_image","position":{"x":100,"y":150},"selected":false,"data":{"specName":"read_image","displayLabel":"read image","description":"Reads a JPEG or PNG image from disk.","inputs":[{"id":"in0","name":"path","type":"string","displayLabel":"file","description":"Path to image.","defaultValue":"sample.png","widget":{"type":"FileInputFromServer","extensions":[".jpg",".jpeg",".png"]}},{"id":"in1","name":"mode","displayLabel":"mode","description":"Colour mode.","defaultValue":"GRAY","widget":{"type":"Dropdown","options":["GRAY","RGB"]}}],"outputs":[{"id":"out0","name":"image","type":"image","displayLabel":"image"}]},"measured":{"width":302,"height":138}},{"id":"1","type":"read_image","position":{"x":550,"y":150},"selected":false,"data":{"specName":"threshold","displayLabel":"threshold","description":"Binarizes by keeping pixels within [lower, upper].","inputs":[{"id":"in0","name":"image","type":"image","displayLabel":"grayscale image","description":"Input image for thresholding."},{"id":"in1","name":"range","type":"tuple2","displayLabel":"Range","description":"Lower and upper threshold bounds.","defaultValue":{"0":0.2,"1":0.8,"histogram":{"type":"grayscale","data":[]}},"widget":{"type":"HistogramRange","min":0,"max":1,"step":0.01}}],"outputs":[{"id":"out0","name":"image","type":"binary image","displayLabel":"image"}]},"measured":{"width":302,"height":212}}],"edges":[{"id":"e0","source":"0","sourceHandle":"out0","target":"1","targetHandle":"in0","selected":false}]}

---
## Step 2 — Connect the client

`ChaldeneClient()` opens a comm channel to the extension and starts tracking
which VP cells are ready. Run this cell after the VP canvas above is visible.

In [2]:
from chaldene import ChaldeneClient

client = ChaldeneClient()
print('Chaldene client ready.')


Chaldene client ready.


---
## Step 3 — Slider

Drag the **lower bound** knob. On release the VP cell re-executes with the
new threshold range via `client.set_input` + `client.run`.

In [3]:
import ipywidgets as widgets

@widgets.interact(
    lower=widgets.FloatSlider(
        min=0.0, max=0.85, step=0.05, value=0.2,
        description='lower:',
        continuous_update=False,
    )
)
def on_threshold_change(lower):
    ids = client.get_ready_cell_ids()
    if not ids:
        return
    client.set_input(ids[-1], '1', 'in1', [lower, 0.9])
    client.run(ids[-1])


interactive(children=(FloatSlider(value=0.2, continuous_update=False, description='lower:', max=0.85, step=0.0…